# SDOF Simulator

Objectives: 

Build a small Python tool that models one spring-mass-damper and shows both how it moves in time and how it responds across frequency.

Set up NumPy/SciPy/matplotlib. Define m, k, c and compute natural frequency and damping ratio.

Simulate the time response to (a) an initial displacement and (b) a harmonic force; plot both.

Compute and plot the FRF magnitude and phase (Bode plot). Overlay curves for several damping ratios.

Add a short markdown note distinguishing receptance, mobility, and accelerance, and verify your numerical natural frequency against sqrt(k/m).


# Imports 

In [7]:
import numpy as np
import matplotlib
matplotlib.use("Agg")         
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.signal import find_peaks

# Parameters

In [8]:
m = 1.0        # mass            [kg]
k = 400.0      # stiffness       [N/m]
c = 2.0        # damping coeff   [N*s/m]    <- try 0 (undamped), 40 (critical), 80 (overdamped)
 
# Derived quantities
wn   = np.sqrt(k / m)                 # undamped natural frequency  [rad/s]
fn   = wn / (2 * np.pi)               # undamped natural frequency  [Hz]
zeta = c / (2 * np.sqrt(k * m))       # damping ratio  []
wd   = wn * np.sqrt(1 - zeta**2) if zeta < 1 else float("nan")   # damped natural freq []

# determine damping regime 
if   zeta == 0: regime = "UNdamped (oscillates forever)"
elif zeta < 1:  regime = "UNDER-damped (decaying oscillation)"
elif zeta == 1: regime = "CRITICALLY-damped (fastest non-oscillating decay)"
else:           regime = "OVER-damped (no oscillation)"
 
print(" SDOF system summary")
print(f"  m = {m} kg,  k = {k} N/m,  c = {c} N*s/m")
print(f"  wn   = {wn:.4f} rad/s   ( fn = {fn:.4f} Hz )")
print(f"  zeta = {zeta:.4f}")
print(f"  wd   = {wd:.4f} rad/s")
print(f"  regime: {regime}")

 SDOF system summary
  m = 1.0 kg,  k = 400.0 N/m,  c = 2.0 N*s/m
  wn   = 20.0000 rad/s   ( fn = 3.1831 Hz )
  zeta = 0.0500
  wd   = 19.9750 rad/s
  regime: UNDER-damped (decaying oscillation)


# Equations of Motion

In [9]:
# numerical solver
def eom(t, y, F):
    x, v = y # unpack state vector into x, x'
    return [v, (F(t) - c * v - k * x) / m] # returns x', x''
 
 
# FREE VIBRATION: initial conditions only, no applied force
x0, v0 = 0.01, 0.0        # 1 cm initial pull, released from rest
 
t_free = np.linspace(0, 8 * (2 * np.pi / wn), 3000)   # ~8 natural periods
 
# numerical solution (works for every regime):
sol = solve_ivp(eom, (t_free[0], t_free[-1]), [x0, v0],
                t_eval=t_free, args=(lambda t: 0.0,), rtol=1e-9, atol=1e-12)
x_num = sol.y[0]
 
# analytical underdamped solution, for comparison / to verify the numerical solver.
# NOTE: the lecture's tidy formula  x0 cos(wd t) + (v0/wd) sin(wd t)  is the
# *small-damping approximation*. The EXACT version keeps a zeta*wn*x0 term:
def free_analytical(t):
    if zeta < 1:
        env = np.exp(-zeta * wn * t)
        return env * (x0 * np.cos(wd * t)
                      + (v0 + zeta * wn * x0) / wd * np.sin(wd * t))
    return np.full_like(t, np.nan)     # (only plotting analytic for underdamped)
 
x_ana = free_analytical(t_free)
envelope = np.sqrt(x0**2 + ((v0 + zeta * wn * x0) / wd)**2) * np.exp(-zeta * wn * t_free) \
           if zeta < 1 else None
 
 
# FORCED VIBRATION: initial conditions + applied force at resonance for max amplitude buildup 
F0            = 1.0
drive_ratio   = 1.0                    # Omega / wn   (1.0 = resonance; try 0.5, 2.0)
Omega         = drive_ratio * wn
t_forced      = np.linspace(0, 12 * (2 * np.pi / wn), 4000)
 
solf = solve_ivp(eom, (t_forced[0], t_forced[-1]), [0.0, 0.0],
                 t_eval=t_forced, args=(lambda t: F0 * np.cos(Omega * t),),
                 rtol=1e-9, atol=1e-12)
x_forced = solf.y[0]

# Frequency Response Function

In [10]:
# ======================================================================
# 3. THE FREQUENCY RESPONSE FUNCTION (FRF)
#    For a harmonic force, the steady-state response is X = H(w) * F.
#    RECEPTANCE (displacement / force):
#            H(w) = 1 / (k - m w^2 + i c w)
#    Two cousins you asked about:
#            MOBILITY   (velocity/force)      =  i*w * H(w)
#            ACCELERANCE(acceleration/force)  = -w^2 * H(w) = (i*w)^2 * H(w)
#    They're the same information viewed through one or two time-derivatives.
# ======================================================================
w = np.linspace(0.01 * wn, 3 * wn, 2000)          # sweep frequency
zetas_to_compare = [0.05, 0.10, 0.25, 0.50, 1.00]  # overlay these damping ratios
 
def receptance(w, zeta_i):
    c_i = zeta_i * 2 * np.sqrt(k * m)             # c that gives this zeta
    return 1.0 / (k - m * w**2 + 1j * c_i * w)
 

# Verification + Plotting

In [11]:
# ======================================================================
# 4. VERIFICATION  --  recover wn and zeta from the SIMULATED free decay.
#    This is the payoff of Lecture 19 section 10 (logarithmic decrement):
#      * peak spacing  -> damped period -> wd -> wn
#      * ratio of peaks n cycles apart -> zeta
#    If our physics/code is right, these should match the inputs above.
# ======================================================================
peaks, _ = find_peaks(x_num)
if len(peaks) >= 3 and zeta < 1:
    tp = t_free[peaks]                 # times of successive peaks
    xp = x_num[peaks]                  # their amplitudes
    Td_meas = np.mean(np.diff(tp))     # measured damped period
    wd_meas = 2 * np.pi / Td_meas
    n = len(peaks) - 1                 # cycles between first and last peak
    delta = (1.0 / n) * np.log(xp[0] / xp[-1])       # logarithmic decrement
    zeta_meas = delta / np.sqrt(4 * np.pi**2 + delta**2)   # exact inversion
    wn_meas = wd_meas / np.sqrt(1 - zeta_meas**2)
    print(" Verification from the simulated decay (log-decrement):")
    print(f"   wn:   input {wn:.4f}  vs recovered {wn_meas:.4f} rad/s")
    print(f"   zeta: input {zeta:.4f}  vs recovered {zeta_meas:.4f}")
    print(f"   (rule of thumb check, zeta ~ 0.11 / n_50%: your system... see plot)")
    print("=" * 60)
 
 
# ======================================================================
# 5. ONE DASHBOARD FIGURE with the four views
# ======================================================================
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("SDOF Simulator  ·  m x'' + c x' + k x = F(t)", fontsize=14, fontweight="bold")
 
# (a) free vibration ---------------------------------------------------
a = ax[0, 0]
a.plot(t_free, x_num * 1000, lw=2, label="numerical (solve_ivp)")
if zeta < 1:
    a.plot(t_free, x_ana * 1000, "--", lw=1.2, label="analytical")
    a.plot(t_free,  envelope * 1000, ":", color="gray", label="decay envelope")
    a.plot(t_free, -envelope * 1000, ":", color="gray")
a.set(title=f"(a) Free vibration from ICs  (zeta = {zeta:.3f})",
      xlabel="time [s]", ylabel="displacement [mm]")
a.legend(fontsize=8); a.grid(alpha=0.3)
 
# (b) forced response --------------------------------------------------
b = ax[0, 1]
b.plot(t_forced, x_forced * 1000, lw=1.5, color="C3")
b.set(title=f"(b) Forced harmonic response  (Omega/wn = {drive_ratio:g})",
      xlabel="time [s]", ylabel="displacement [mm]")
b.grid(alpha=0.3)
 
# (c) FRF magnitude ----------------------------------------------------
cax = ax[1, 0]
for z in zetas_to_compare:
    cax.plot(w / wn, np.abs(receptance(w, z)), label=f"zeta = {z:.2f}")
cax.axvline(1.0, color="gray", ls=":", lw=1)
cax.set(title="(c) FRF magnitude  |X/F|  (receptance)",
        xlabel="frequency ratio  Omega / wn", ylabel="|H|  [m/N]", yscale="log")
cax.legend(fontsize=8); cax.grid(alpha=0.3, which="both")
 
# (d) FRF phase --------------------------------------------------------
d = ax[1, 1]
for z in zetas_to_compare:
    d.plot(w / wn, np.degrees(np.angle(receptance(w, z))), label=f"zeta = {z:.2f}")
d.axvline(1.0, color="gray", ls=":", lw=1)
d.set(title="(d) FRF phase", xlabel="frequency ratio  Omega / wn",
      ylabel="phase [deg]")
d.set_yticks([0, -45, -90, -135, -180])
d.legend(fontsize=8); d.grid(alpha=0.3)
 
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig("sdof_dashboard.png", dpi=130)
print("Saved figure -> sdof_dashboard.png")

 Verification from the simulated decay (log-decrement):
   wn:   input 20.0000  vs recovered 20.0006 rad/s
   zeta: input 0.0500  vs recovered 0.0500
   (rule of thumb check, zeta ~ 0.11 / n_50%: your system... see plot)
Saved figure -> sdof_dashboard.png
